In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch import nn, optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
DATA_DIR = "/kaggle/input/plantdisease/PlantVillage"

print("Number of classes:", len(os.listdir(DATA_DIR)))
print("Classes:\n", os.listdir(DATA_DIR))


In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])


In [ ]:
dataset = datasets.ImageFolder(
    root=DATA_DIR,
    transform=transform
)

print("Total images:", len(dataset))
print("Number of classes:", len(dataset.classes))


In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = SimpleCNN(num_classes=len(dataset.classes)).to(device)
print(model)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
def train_model(model, train_loader, val_loader, epochs=5):
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = 100 * correct / total
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Train Acc: {train_acc:.2f}% "
              f"Val Acc: {val_acc:.2f}%")

train_model(model, train_loader, val_loader, epochs=5)


In [ ]:
import matplotlib.pyplot as plt
import torch

# ---------- CLEAN LABEL MAP ----------
label_map = {
    "Tomato_Bacterial_spot": "Bacterial Spot",
    "Tomato_Tomato_YellowLeaf_Curl_Virus": "Yellow Leaf Curl",
    "Tomato_Target_Spot": "Target Spot",
    "Tomato_Leaf_Mold": "Leaf Mold",
    "Tomato_Late_blight": "Late Blight",
    "Tomato_Early_blight": "Early Blight",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Spider Mites",
    "Tomato_healthy": "Healthy"
}

def clean_label(raw_label):
    return label_map.get(raw_label, raw_label.replace("_", " "))


def show_predictions_ieee(model, loader):
    model.eval()
    images, labels = next(iter(loader))
    images = images.to(device)

    outputs = model(images)
    _, preds = torch.max(outputs, 1)

    images = images.cpu()

    fig, axes = plt.subplots(2, 4, figsize=(10, 5))

    for i, ax in enumerate(axes.flat):

        img = images[i].permute(1, 2, 0)
        img = img * 0.5 + 0.5  # unnormalize if needed

        ax.imshow(img)
        ax.axis("off")

        raw_name = classes[preds[i]]
        clean_name = clean_label(raw_name)

        # Subfigure label (a), (b), (c)...
        sublabel = f"({chr(97+i)}) {clean_name}"

        ax.set_title(sublabel, fontsize=10, pad=4)

        # Thin border (professional look)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)

    plt.subplots_adjust(wspace=0.05, hspace=0.25)

    # ---------- SAVE HIGH-QUALITY ----------
    plt.savefig(
        "prediction_examples_ieee.png",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02
    )

    plt.show()


show_predictions_ieee(model, val_loader)